# SmartParking AI — шаблон выполнения задания

Учебный ML-проект: по кадру с камеры найти номер, прочитать его и решить, пускать автомобиль, отказать или отправить кадр на ручную проверку.

Источник требований: `SmartParking_AI_план_реализации.txt`. Этот ноутбук — рабочий шаблон. Ячейки с инфраструктурой можно запускать сразу. Ячейки обучения и скачивания датасетов закрыты флагами `DOWNLOAD_DATA` и `RUN_TRAINING`, чтобы случайный запуск не качал данные и не занимал GPU.

**Что сдаётся по плану:** обученные модели, измеримые эксперименты, работающий MVP, бизнес-модель и защита перед преподавателем-инвестором.

**Ориентиры качества (не гарантия):** mAP@50 ≥ 0.90 на детекции и высокий Exact Match на качественных дневных кадрах. Итоговые цифры вносить только после реального запуска.

**Стек раннего MVP:** Python 3.11+, Ultralytics YOLO, PaddleOCR, OpenCV, Streamlit, SQLite. Полный стек из плана: PyTorch, YOLO11n/YOLO11s, CRNN + CTC, Albumentations, Pandas, Matplotlib, Weights & Biases.

Заполните карточку команды в следующей ячейке до защиты.


In [ ]:
# Карточка команды — впишите ФИО. Части задания фиксированы ниже.
TEAM = [
    {
        "part": 1,
        "name": "ФИО",
        "role": "Данные и EDA",
        "notebook": "Раздел 1",
        "delivers": "выборки train/val/test, 5–8 графиков, выводы по данным",
    },
    {
        "part": 2,
        "name": "ФИО",
        "role": "Детекция YOLO",
        "notebook": "Раздел 2",
        "delivers": "веса детектора и заполненный DETECTION_EXPERIMENTS",
    },
    {
        "part": 3,
        "name": "ФИО",
        "role": "OCR: PaddleOCR и CRNN",
        "notebook": "Раздел 3",
        "delivers": "сравнение OCR и заполненный OCR_EXPERIMENTS",
    },
    {
        "part": 4,
        "name": "ФИО",
        "role": "Пайплайн и демо",
        "notebook": "Раздел 4, app.py",
        "delivers": "recognize_image, видео, Streamlit, сценарии А/Б/В",
    },
    {
        "part": 5,
        "name": "ФИО",
        "role": "Оценка, бизнес и защита",
        "notebook": "Разделы 5–6",
        "delivers": "SYSTEM_EVAL, конкуренты, экономика, питч",
    },
]
WEEKS = "4–6"
NOTES = "Каждый отвечает за свою часть и на защите может объяснить общий пайплайн."


## Кто за что отвечает

Пять частей совпадают с разделами этого ноутбука. Чужие ячейки не переписывать: результат передаётся файлами, весами и заполненными таблицами.

| Часть | Раздел | Сдаёт | Когда можно начинать |
|---|---|---|---|
| 1. Данные | Раздел 1 | Датасеты, проверка битых файлов и дублей, 5–8 графиков EDA, сплит без утечки кадров, каталоги YOLO и пары для OCR | Сразу. Это блокирует части 2 и 3 |
| 2. Детекция | Раздел 2 | Обученный YOLO11n, сравнение аугментаций, по возможности 960 px и YOLO11s, метрики в `DETECTION_EXPERIMENTS`, примеры удачных и ошибочных детекций | После сплита части 1. Горизонтальное отражение номеров не использовать |
| 3. OCR | Раздел 3 | Рабочий PaddleOCR, обученная CRNN, CER / Exact Match в `OCR_EXPERIMENTS`, разбор путаниц символов | Параллельно с частью 2, как только готовы кропы номеров. Если CRNN не успевает, MVP идёт на PaddleOCR |
| 4. Продукт | Раздел 4 и `app.py` | `recognize_image`, агрегация кадров видео, белый список SQLite, Streamlit, три сценария в `DEMO_SCENARIOS` | Интерфейс и базу можно собирать на заглушках. Живое демо — после весов частей 2 и 3 |
| 5. Защита | Разделы 5–6 | Метрики доступа в `SYSTEM_EVAL`, разбор ошибок, 3 конкурента, расходы, 10–12 слайдов, спич на 5–8 минут и резервное видео | Конкурентов и экономику считать сразу. Цифры качества — только после прогонов частей 2–4 |

Общее для всех: понимать цепочку «кадр → YOLO → OCR → решение», не коммитить датасеты и чужие видео, не вписывать метрики до реального запуска. Репетицию защиты проходит вся команда, сводит её часть 5.


## 0. Окружение и каталоги

Структура из плана:

```
data/raw  data/processed  data/demo
notebooks  src  configs  models  reports  tests
```

Большие датасеты, видео с персональными данными и ключи в git не коммитить.


In [ ]:
from __future__ import annotations

import random
import re
import sqlite3
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np

try:
    import pandas as pd
except ImportError as exc:
    raise ImportError(
        "Для таблиц экспериментов нужен pandas: pip install pandas numpy"
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Корень репозитория: ноутбук лежит в notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DIRS = {
    "raw": ROOT / "data" / "raw",
    "processed": ROOT / "data" / "processed",
    "demo": ROOT / "data" / "demo",
    "detection": ROOT / "data" / "processed" / "detection",
    "ocr": ROOT / "data" / "processed" / "ocr",
    "configs": ROOT / "configs",
    "models": ROOT / "models",
    "reports": ROOT / "reports",
    "runs": ROOT / "runs",
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

# Включайте осознанно: скачивание и обучение не должны стартовать при «Run All».
DOWNLOAD_DATA = False
RUN_TRAINING = False
RUN_OCR_TRAINING = False

try:
    import torch

    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TORCH_VERSION = torch.__version__
except ImportError:
    torch = None
    DEVICE = "cpu"
    TORCH_VERSION = "не установлен"

print(f"ROOT = {ROOT}")
print(f"PyTorch = {TORCH_VERSION}, device = {DEVICE}")
print("Каталоги созданы:", ", ".join(p.name for p in DIRS.values()))


## 1. Неделя 1. Данные и EDA

**Цель:** изучить датасеты и подготовить выборки без утечки почти одинаковых кадров одной машины.

| Датасет | Зачем | Ссылка |
|---|---|---|
| Car Plate Detection | детекция российских номеров | https://huggingface.co/datasets/AY000554/Car_plate_detecting_dataset |
| Car Plate OCR | картинка номера + текст | https://huggingface.co/datasets/AY000554/Car_plate_OCR_dataset |
| UFPR-ALPR | дополнительный тест с камер | https://github.com/raysonlaroca/ufpr-alpr-dataset |
| CCPD | устойчивость детекции, не основной источник российского OCR | https://github.com/detectRecog/CCPD |

Перед загрузкой проверьте лицензию и правила публикации кадров. Свой тест: 100–300 кадров в условиях, близких к парковке. Гиперпараметры по финальному test не подбирать.

Сплит по умолчанию: **70% / 15% / 15%**.

Чеклист недели:

- [ ] скачать датасеты и описать формат разметки
- [ ] убрать битые файлы и дубликаты
- [ ] посчитать число изображений и распределение размеров
- [ ] оценить размер номера относительно кадра
- [ ] частоты символов, длину строки, коды регионов
- [ ] яркость, размытие, сложные ракурсы
- [ ] train / val / test без утечек
- [ ] 5–8 графиков и короткие выводы под каждым


In [ ]:
DETECTION_DATASET = "AY000554/Car_plate_detecting_dataset"
OCR_DATASET = "AY000554/Car_plate_OCR_dataset"
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}


def download_hf_dataset(repo_id: str, dest: Path) -> Path:
    """Скачивает датасет с Hugging Face. Вызывайте только при DOWNLOAD_DATA = True."""
    dest.mkdir(parents=True, exist_ok=True)
    # TODO: выбрать способ загрузки после проверки лицензии.
    # Вариант: huggingface_hub.snapshot_download(repo_id, repo_type="dataset", local_dir=dest)
    raise NotImplementedError(f"Опишите загрузку {repo_id} в {dest}")


if DOWNLOAD_DATA:
    download_hf_dataset(DETECTION_DATASET, DIRS["raw"] / "detection")
    download_hf_dataset(OCR_DATASET, DIRS["raw"] / "ocr")
else:
    print("DOWNLOAD_DATA = False. Датасеты не скачиваются.")
    print("Положите сырые данные в", DIRS["raw"])


In [ ]:
def list_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.suffix.lower() in exts)


raw_images = list_images(DIRS["raw"])
print(f"Изображений в data/raw: {len(raw_images)}")
if not raw_images:
    print("EDA ниже заполнится, когда в data/raw появятся файлы.")


### 1.1. Проверки качества файлов

Заполните функции и сохраните краткий отчёт: сколько файлов удалено, сколько дублей, какой формат разметки (YOLO txt, COCO, CSV с текстом номера).


In [ ]:
def is_readable_image(path: Path) -> bool:
    """True, если файл открывается как изображение."""
    # TODO: cv2.imread / PIL.Image.open и проверка, что массив не пустой
    raise NotImplementedError


def file_hash(path: Path) -> str:
    """Хеш содержимого, чтобы ловить точные дубликаты."""
    # TODO: hashlib.md5 или sha1 по байтам файла
    raise NotImplementedError


def find_duplicates(paths: list[Path]) -> dict[str, list[Path]]:
    """Группы путей с одинаковым хешем. Одиночные файлы не возвращать."""
    # TODO
    raise NotImplementedError


# После реализации раскомментируйте и допишите вывод:
# broken = [p for p in raw_images if not is_readable_image(p)]
# duplicates = find_duplicates(raw_images)
# print("битые:", len(broken), "групп дублей:", len(duplicates))
print("Функции проверки объявлены. Реализуйте TODO и снимите комментарий с прогона.")


### 1.2. Графики EDA

Сделайте 5–8 графиков и под каждым одну-две фразы: что видно и как это влияет на обучение.

Идеи графиков:

1. Гистограмма ширины и высоты кадра.
2. Гистограмма площади номера относительно кадра.
3. Частоты символов и длины строки.
4. Распределение кодов региона.
5. Яркость (средняя по кадру) и оценка размытия (дисперсия Лапласиана).
6. Примеры сложных ракурсов — сетка изображений, не только гистограмма.

Горизонтальное отражение номеров в аугментациях не использовать: оно ломает порядок символов.


In [ ]:
# TODO: импортировать matplotlib (и cv2, pandas — когда понадобятся) и построить графики.
# Сохраняйте фигуры в DIRS["reports"], например reports/eda_image_sizes.png
#
# import matplotlib.pyplot as plt
#
# def plot_image_sizes(paths: list[Path], save_to: Path) -> None:
#     ...
#
# Вывод под графиком (заполнить после просмотра):
EDA_CONCLUSIONS = {
    "sizes": "",
    "plate_scale": "",
    "symbols": "",
    "regions": "",
    "quality": "",
    "limitations": "",
}
EDA_CONCLUSIONS


### 1.3. Разделение train / val / test

Ключ сплита — автомобиль или сцена, а не отдельный кадр. Почти одинаковые кадры одной машины не должны оказаться и в train, и в test.

На выходе для детекции нужна структура YOLO:

```
data/processed/detection/{train,val,test}/images
data/processed/detection/{train,val,test}/labels
```

Для OCR: пары «изображение номера + строка» в `data/processed/ocr`, тоже три сплита.


In [ ]:
def split_ids(group_ids: list[str], split: dict[str, float], seed: int = SEED):
    """Делит уникальные id групп на train/val/test.

    group_ids — идентификаторы машин/сцен, не пути к почти одинаковым кадрам.
    Доли в split должны суммироваться в 1.
    """
    if abs(sum(split.values()) - 1.0) > 1e-6:
        raise ValueError("Доли сплита должны давать 1")
    ids = sorted(set(group_ids))
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_train = int(round(n * split["train"]))
    n_val = int(round(n * split["val"]))
    # Остаток отдаём test, чтобы сумма была равна n
    parts = {
        "train": ids[:n_train],
        "val": ids[n_train : n_train + n_val],
        "test": ids[n_train + n_val :],
    }
    assert sum(len(v) for v in parts.values()) == n
    return parts


# Пример на синтетических id. Замените их реальными ключами групп из датасета.
_demo = split_ids([f"car_{i:04d}" for i in range(100)], SPLIT)
{k: len(v) for k, v in _demo.items()}


## 2. Неделя 2. Детекция YOLO

**Цель:** находить номер на фотографии машины.

Сравнение, которое нужно зафиксировать:

1. YOLO без аугментаций и YOLO с аугментациями (яркость, шум, размытие, небольшой поворот).
2. YOLO11n и YOLO11s — если хватает ресурсов.
3. Вход 640 px и 960 px.

Метрики: Precision, Recall, mAP@50, mAP@50–95, latency на один кадр. Сохранить удачные и ошибочные детекции.

Не отражать номера по горизонтали.


In [ ]:
PLATES_YAML = DIRS["configs"] / "plates.yaml"
PLATES_YAML.write_text(
    """path: data/processed/detection
train: train/images
val: val/images
test: test/images
nc: 1
names:
  0: license_plate
""",
    encoding="utf-8",
)
print("Записан", PLATES_YAML)


In [ ]:
def train_detector(
    weights: str = "yolo11n.pt",
    imgsz: int = 640,
    epochs: int = 50,
    batch: int = 16,
    name: str = "yolo11n_baseline",
):
    """Обучение детектора. Запускать только при RUN_TRAINING = True и готовом data.yaml."""
    from ultralytics import YOLO

    model = YOLO(weights)
    model.train(
        data=str(PLATES_YAML),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        patience=10,
        seed=SEED,
        project=str(DIRS["runs"] / "detector"),
        name=name,
        # TODO: явно отключить fliplr, когда будете включать свои аугментации:
        # fliplr=0.0,
    )
    metrics = model.val(data=str(PLATES_YAML), split="test")
    return metrics.results_dict


if RUN_TRAINING:
    baseline_metrics = train_detector()
    print(baseline_metrics)
else:
    print("RUN_TRAINING = False. Обучение детектора пропущено.")
    print("Когда данные готовы, поставьте флаг True и сравните эксперименты из плана.")


### Журнал экспериментов детекции

Одну строку — один реальный запуск. Пустые строки не заполняйте «ожидаемыми» цифрами.


In [ ]:
DETECTION_EXPERIMENTS = pd.DataFrame(
    [
        {
            "name": "yolo11n_baseline",
            "weights": "yolo11n.pt",
            "imgsz": 640,
            "augment": False,
            "hypothesis": "Предобученная YOLO11n без доп. аугментаций уже находит номер на дневных кадрах.",
            "precision": None,
            "recall": None,
            "map50": None,
            "map50_95": None,
            "latency_ms": None,
            "interpretation": "",
        },
        {
            "name": "yolo11n_aug",
            "weights": "yolo11n.pt",
            "imgsz": 640,
            "augment": True,
            "hypothesis": "Яркость, шум, размытие и небольшой поворот поднимут recall на сложных кадрах.",
            "precision": None,
            "recall": None,
            "map50": None,
            "map50_95": None,
            "latency_ms": None,
            "interpretation": "",
        },
        {
            "name": "yolo11n_960",
            "weights": "yolo11n.pt",
            "imgsz": 960,
            "augment": True,
            "hypothesis": "Больший вход поможет находить мелкие номера.",
            "precision": None,
            "recall": None,
            "map50": None,
            "map50_95": None,
            "latency_ms": None,
            "interpretation": "",
        },
        {
            "name": "yolo11s",
            "weights": "yolo11s.pt",
            "imgsz": 640,
            "augment": True,
            "hypothesis": "YOLO11s точнее YOLO11n при приемлемой задержке.",
            "precision": None,
            "recall": None,
            "map50": None,
            "map50_95": None,
            "latency_ms": None,
            "interpretation": "",
        },
    ]
)
DETECTION_EXPERIMENTS


## 3. Неделя 3. OCR

**Цель:** превратить вырезанный номер в текст и честно сравнить два подхода.

- Baseline: PaddleOCR.
- Своя модель: CNN → BiLSTM → Linear → CTC Loss.

Стартовые параметры CRNN из плана:

| Параметр | Значение |
|---|---|
| Вход | около 32×160, пропорции сохранять, дальше padding |
| Оптимизатор | AdamW |
| Learning rate | 0.001 |
| Batch size | 32–64 |
| Эпохи | 20–40, ранняя остановка |
| Loss | CTC |

Метрики: Character Accuracy, CER, Exact Match. Отдельно разобрать путаницы 0/O, 8/B и кадры с низкой контрастностью.

Если CRNN не успевает к защите, MVP собирается на PaddleOCR, а CRNN остаётся отдельным экспериментом.


In [ ]:
# Латинские буквы, допустимые на типовом российском легковом номере.
# Кириллические двойники приводятся к этому алфавиту до проверки формата.
PLATE_ALPHABET = "0123456789ABEKMHOPCTYX"
CTC_BLANK = 0  # индекс blank в выходе CTC; символы алфавита занимают 1..len(PLATE_ALPHABET)

CYR_TO_LAT = str.maketrans(
    {
        "А": "A",
        "В": "B",
        "Е": "E",
        "К": "K",
        "М": "M",
        "Н": "H",
        "О": "O",
        "Р": "P",
        "С": "C",
        "Т": "T",
        "У": "Y",
        "Х": "X",
        "а": "A",
        "в": "B",
        "е": "E",
        "к": "K",
        "м": "M",
        "н": "H",
        "о": "O",
        "р": "P",
        "с": "C",
        "т": "T",
        "у": "Y",
        "х": "X",
    }
)


def normalize_plate(text: str) -> str:
    """Убирает пробелы и приводит похожие кириллические буквы к латинице."""
    cleaned = "".join(text.upper().split())
    return cleaned.translate(CYR_TO_LAT)


# Шаблон из плана. Он не покрывает все типы российских номеров.
PLATE_PATTERN = re.compile(r"^[ABEKMHOPCTYX]\d{3}[ABEKMHOPCTYX]{2}\d{2,3}$")


def valid_plate(text: str) -> bool:
    return bool(PLATE_PATTERN.fullmatch(normalize_plate(text)))


samples = ["а123вс77", "A123BC777", "A 123 BC 77", "123ABC77", "O000OO00"]
[(s, normalize_plate(s), valid_plate(s)) for s in samples]


In [ ]:
def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        for j, cb in enumerate(b, start=1):
            ins = cur[j - 1] + 1
            delete = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            cur.append(min(ins, delete, sub))
        prev = cur
    return prev[-1]


def cer(pred: str, target: str) -> float:
    """Character Error Rate. Ноль символов в target даёт 0.0 только при пустом pred."""
    target_n = normalize_plate(target)
    pred_n = normalize_plate(pred)
    if not target_n:
        return 0.0 if not pred_n else 1.0
    return levenshtein(pred_n, target_n) / len(target_n)


def char_accuracy(pred: str, target: str) -> float:
    return max(0.0, 1.0 - cer(pred, target))


def exact_match(pred: str, target: str) -> bool:
    return normalize_plate(pred) == normalize_plate(target)


def ocr_report(pairs: list[tuple[str, str]]) -> dict[str, float]:
    """pairs: список (предсказание, эталон)."""
    if not pairs:
        return {"cer": float("nan"), "char_accuracy": float("nan"), "exact_match": float("nan"), "n": 0}
    cers = [cer(p, t) for p, t in pairs]
    matches = [exact_match(p, t) for p, t in pairs]
    return {
        "n": len(pairs),
        "cer": float(np.mean(cers)),
        "char_accuracy": float(np.mean([max(0.0, 1.0 - c) for c in cers])),
        "exact_match": float(np.mean(matches)),
    }


demo_pairs = [("A123BC77", "А123ВС77"), ("A123BC78", "A123BC77"), ("B888BK99", "B888BK99")]
ocr_report(demo_pairs)


### 3.1. PaddleOCR baseline

Подключите модель, прогоните validation и test, сложите строки в `OCR_EXPERIMENTS`. На этом шаге не подгоняйте алфавит по test.


In [ ]:
def read_plate_paddle(image_path: Path) -> tuple[str, float]:
    """Возвращает (текст, уверенность) для уже вырезанного номера."""
    # TODO:
    # from paddleocr import PaddleOCR
    # ocr = PaddleOCR(use_angle_cls=True, lang="en")  # язык уточнить по фактической модели
    # result = ocr.ocr(str(image_path), cls=True)
    # разобрать result в строку и среднюю уверенность
    raise NotImplementedError


print("PaddleOCR подключается в read_plate_paddle, когда установлен пакет и готовы кропы номеров.")


### 3.2. CRNN

Каркас ниже соответствует схеме из плана и входу 1×32×160. Это стартовая архитектура, её можно менять, но тогда запишите, что изменилось и зачем.

CTC ждёт логарифмы вероятностей формы `(T, N, C)`, где `C` включает blank.


In [ ]:
def build_crnn(num_classes: int):
    if torch is None:
        raise ImportError("Для CRNN нужен PyTorch")
    import torch.nn as nn

    class CRNN(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.cnn = nn.Sequential(
                nn.Conv2d(1, 64, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),  # 16 x 80
                nn.Conv2d(64, 128, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),  # 8 x 40
                nn.Conv2d(128, 256, 3, padding=1),
                nn.BatchNorm2d(256),
                nn.ReLU(inplace=True),
                nn.MaxPool2d((2, 1), (2, 1)),  # 4 x 40
                nn.Conv2d(256, 256, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d((2, 1), (2, 1)),  # 2 x 40
                nn.Conv2d(256, 256, 3, padding=1),
                nn.BatchNorm2d(256),
                nn.ReLU(inplace=True),
                nn.MaxPool2d((2, 1), (2, 1)),  # 1 x 40
            )
            self.rnn = nn.LSTM(
                256,
                256,
                num_layers=2,
                bidirectional=True,
                batch_first=True,
            )
            self.fc = nn.Linear(256 * 2, num_classes)

        def forward(self, x):
            # x: (N, 1, 32, 160)
            feat = self.cnn(x).squeeze(2).permute(0, 2, 1)  # (N, T, 256)
            seq, _ = self.rnn(feat)
            logits = self.fc(seq)  # (N, T, C)
            return logits.permute(1, 0, 2)  # (T, N, C)

    return CRNN()


if torch is not None:
    _crnn = build_crnn(num_classes=1 + len(PLATE_ALPHABET)).to(DEVICE)
    _dummy = torch.zeros(2, 1, 32, 160, device=DEVICE)
    with torch.no_grad():
        _out = _crnn(_dummy)
    print("CRNN output:", tuple(_out.shape), "device:", DEVICE)
else:
    print("PyTorch не установлен, проверка формы CRNN пропущена.")


In [ ]:
def train_crnn(model, train_loader, val_loader, epochs: int = 30, lr: float = 1e-3):
    """Цикл обучения с CTC, AdamW и ранней остановкой.

    TODO:
    - подготовить Dataset: серое изображение 32x160 и целевая строка в индексах алфавита;
    - blank имеет индекс CTC_BLANK, символы — с 1;
    - Criterion: torch.nn.CTCLoss(blank=CTC_BLANK, zero_infinity=True);
    - логиты перед loss пропустить через log_softmax по классу;
    - сохранять лучшие веса в models/crnn_best.pt по val CER;
    - останавливаться, если val CER не улучшается несколько эпох.
    """
    if not RUN_OCR_TRAINING:
        print("RUN_OCR_TRAINING = False. Цикл обучения CRNN не запущен.")
        return None
    raise NotImplementedError("Соберите цикл обучения по TODO в docstring")


In [ ]:
OCR_EXPERIMENTS = pd.DataFrame(
    [
        {
            "name": "paddleocr",
            "hypothesis": "Готовый PaddleOCR даёт рабочий baseline без своего обучения.",
            "cer": None,
            "char_accuracy": None,
            "exact_match": None,
            "notes": "",
        },
        {
            "name": "crnn_ctc",
            "hypothesis": "CRNN, обученная на кропах российских номеров, обгонит baseline по Exact Match.",
            "cer": None,
            "char_accuracy": None,
            "exact_match": None,
            "notes": "",
        },
    ]
)

# TODO: таблица частых подмен символов, например 0↔O, 8↔B.
CONFUSIONS = pd.DataFrame(columns=["target", "pred", "count", "example"])
OCR_EXPERIMENTS


## 4. Неделя 4. Пайплайн и решение о доступе

Порядок:

`кадр → YOLO → вырезать и выровнять номер → OCR → нормализация → проверка шаблона → голосование по кадрам видео → порог уверенности → белый список SQLite`

На защите шлагбаум не нужен. Достаточно решения в интерфейсе.

Три обязательных сценария:

- **А.** Номер из белого списка → разрешить.
- **Б.** Уверенно прочитанный, но неизвестный номер → отказать.
- **В.** Низкая уверенность или номер не проходит шаблон → ручная проверка.

Одно слабое предсказание не должно открывать доступ.


In [ ]:
CONF_THRESHOLD = 0.85


@dataclass
class PlateRead:
    text: str
    confidence: float


def aggregate_reads(reads: list[PlateRead]) -> PlateRead:
    """Голосование по нескольким кадрам одного автомобиля.

    Берётся нормализованный текст с наибольшей суммарной уверенностью.
    Уверенность результата — средняя уверенность голосов за этот текст.
    """
    if not reads:
        return PlateRead(text="", confidence=0.0)
    scores: dict[str, list[float]] = {}
    for item in reads:
        key = normalize_plate(item.text)
        scores.setdefault(key, []).append(float(item.confidence))
    best = max(scores, key=lambda key: sum(scores[key]))
    votes = scores[best]
    return PlateRead(text=best, confidence=sum(votes) / len(votes))


def decide_access(plate: str, confidence: float, whitelist: set[str], threshold: float = CONF_THRESHOLD) -> str:
    """allow | deny | manual_review."""
    text = normalize_plate(plate)
    if confidence < threshold or not valid_plate(text):
        return "manual_review"
    if text in {normalize_plate(item) for item in whitelist}:
        return "allow"
    return "deny"


whitelist_demo = {"A123BC77", "B888BK199"}
frames = [
    PlateRead("а123вс77", 0.91),
    PlateRead("A123BC77", 0.87),
    PlateRead("A1238C77", 0.42),
]
agg = aggregate_reads(frames)
print(agg, "→", decide_access(agg.text, agg.confidence, whitelist_demo))
print("неизвестный:", decide_access("K555MM77", 0.93, whitelist_demo))
print("сомнительный:", decide_access("K555MM77", 0.40, whitelist_demo))


In [ ]:
DB_PATH = DIRS["processed"] / "access.sqlite"


def connect(db_path: Path = DB_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS whitelist (
            plate TEXT PRIMARY KEY,
            owner TEXT,
            note TEXT,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
        """
    )
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT DEFAULT CURRENT_TIMESTAMP,
            plate TEXT,
            confidence REAL,
            decision TEXT,
            source TEXT
        )
        """
    )
    conn.commit()
    return conn


def add_whitelist(plate: str, owner: str = "", note: str = "", db_path: Path = DB_PATH) -> None:
    text = normalize_plate(plate)
    if not valid_plate(text):
        raise ValueError(f"Номер не проходит шаблон: {plate!r}")
    with connect(db_path) as conn:
        conn.execute(
            "INSERT OR REPLACE INTO whitelist (plate, owner, note) VALUES (?, ?, ?)",
            (text, owner, note),
        )


def load_whitelist(db_path: Path = DB_PATH) -> set[str]:
    with connect(db_path) as conn:
        rows = conn.execute("SELECT plate FROM whitelist").fetchall()
    return {row["plate"] for row in rows}


def log_event(plate: str, confidence: float, decision: str, source: str, db_path: Path = DB_PATH) -> None:
    with connect(db_path) as conn:
        conn.execute(
            "INSERT INTO events (plate, confidence, decision, source) VALUES (?, ?, ?, ?)",
            (normalize_plate(plate), confidence, decision, source),
        )


add_whitelist("A123BC77", owner="демо-резидент")
print("белый список:", sorted(load_whitelist()))
print("база:", DB_PATH)


### 4.1. Сквозной прогон одного изображения

Соберите функцию, когда появятся веса детектора и OCR. Для видео: трек или простая группировка соседних кадров одного авто, затем `aggregate_reads`.

Интерфейс Streamlit по плану живёт в `app.py`, не в этом ноутбуке. Здесь достаточно показать три сценария на фото или на нескольких кадрах.


In [ ]:
def recognize_image(image_path: Path) -> dict:
    """Полный шаг: детекция, кроп, OCR, решение.

    TODO:
    1. Прочитать кадр через OpenCV.
    2. Найти номер моделью YOLO, взять бокс с максимальной уверенностью.
    3. Вырезать и при необходимости выровнять область номера.
    4. Прочитать текст (PaddleOCR или CRNN) и уверенность.
    5. Вернуть text, confidence, decision, latency_ms, bbox.
    """
    started = time.perf_counter()
    # result = ...
    elapsed_ms = (time.perf_counter() - started) * 1000
    return {
        "path": str(image_path),
        "text": "",
        "confidence": 0.0,
        "decision": "manual_review",
        "latency_ms": elapsed_ms,
        "bbox": None,
        "todo": "подключить YOLO и OCR",
    }


# Пример вызова, когда появится файл:
# recognize_image(DIRS["demo"] / "known_plate.jpg")
print("recognize_image — точка сборки MVP. Реализуйте TODO перед демо.")


### 4.2. Три демонстрационных сценария

Положите файлы в `data/demo/` и прогоните ячейку. В таблицу запишите фактическое решение системы.


In [ ]:
DEMO_SCENARIOS = pd.DataFrame(
    [
        {
            "id": "A",
            "file": "data/demo/known_plate.jpg",
            "expect": "allow",
            "got": "",
            "plate": "",
            "confidence": None,
            "comment": "Номер есть в белом списке",
        },
        {
            "id": "B",
            "file": "data/demo/unknown_plate.jpg",
            "expect": "deny",
            "got": "",
            "plate": "",
            "confidence": None,
            "comment": "Номер прочитан уверенно и отсутствует в списке",
        },
        {
            "id": "C",
            "file": "data/demo/hard_frame.jpg",
            "expect": "manual_review",
            "got": "",
            "plate": "",
            "confidence": None,
            "comment": "Смазанный, тёмный или частичный номер",
        },
    ]
)
DEMO_SCENARIOS


## 5. Неделя 5. Оценка системы

Считать не только качество детекции и OCR по отдельности, но и бизнес-решение на отложенном тесте.

| Метрика | Смысл |
|---|---|
| Доля верных решений | allow / deny / manual_review совпали с разметкой сценария |
| Доля ложных разрешений | система пустила того, кого пускать нельзя. Для доступа это главная ошибка |
| Доля ручной проверки | сколько кадров ушло оператору |
| Задержка | время от кадра до решения на вашем оборудовании |

Отдельные сравнения:

- исходный датасет и собственное тестовое видео;
- один кадр и агрегация нескольких кадров.

Оформление графиков — в `reports/`. Если пользуетесь Weights & Biases, ссылку на run запишите в ячейку ниже. Цифры только с реальных прогонов.


In [ ]:
def access_metrics(rows: list[dict]) -> dict[str, float]:
    """rows: элементы с ключами y_true и y_pred из {allow, deny, manual_review}."""
    if not rows:
        return {}
    n = len(rows)
    correct = sum(r["y_true"] == r["y_pred"] for r in rows)
    false_allow = sum(r["y_pred"] == "allow" and r["y_true"] != "allow" for r in rows)
    manual = sum(r["y_pred"] == "manual_review" for r in rows)
    return {
        "n": n,
        "decision_accuracy": correct / n,
        "false_allow_rate": false_allow / n,
        "manual_review_rate": manual / n,
    }


# Проверка определений на игрушечных строках, не на датасете.
access_metrics(
    [
        {"y_true": "allow", "y_pred": "allow"},
        {"y_true": "deny", "y_pred": "allow"},
        {"y_true": "deny", "y_pred": "manual_review"},
        {"y_true": "manual_review", "y_pred": "manual_review"},
    ]
)


In [ ]:
SYSTEM_EVAL = pd.DataFrame(
    [
        {
            "setup": "один кадр, исходный test",
            "decision_accuracy": None,
            "false_allow_rate": None,
            "manual_review_rate": None,
            "latency_ms": None,
            "hardware": "",
            "interpretation": "",
        },
        {
            "setup": "агрегация кадров, исходный test",
            "decision_accuracy": None,
            "false_allow_rate": None,
            "manual_review_rate": None,
            "latency_ms": None,
            "hardware": "",
            "interpretation": "",
        },
        {
            "setup": "агрегация кадров, своё видео",
            "decision_accuracy": None,
            "false_allow_rate": None,
            "manual_review_rate": None,
            "latency_ms": None,
            "hardware": "",
            "interpretation": "",
        },
    ]
)

WANDB_URL = ""  # ссылка на проект или конкретный run
SYSTEM_EVAL


### Разбор ошибок

Кратко ответьте по фактам прогона:

- Где детектор теряет номер (мелкий номер, ночь, сильный угол)?
- Какие символы путает OCR?
- Есть ли ложные разрешения, и на каких кадрах?
- Что вы меняете в первую очередь, если останется ещё одна итерация?


In [ ]:
ERROR_ANALYSIS = {
    "detector_misses": "",
    "ocr_confusions": "",
    "false_allows": "",
    "next_iteration": "",
}
ERROR_ANALYSIS


## 6. Неделя 6. Бизнес-модель для питча

Продукт продаётся как экономия времени на контроле въезда, а не как «нейросеть».

Ниже — **учебный пример из плана, не рыночный прогноз**. Подставьте свои допущения и отдельно посчитайте расходы: установка, мини-ПК или сервер, поддержка, разработка, продажи, налоги, лицензии. Выручка не равна прибыли.

Формула из плана: 20 объектов × 2 камеры × 3 000 ₽ в месяц.

Для слайдов нужно минимум три конкурента: цена, функции, локальная работа, интеграции. Плюс ответ, что в текущих решениях дорого или неудобно маленькой парковке.


In [ ]:
objects = 20
cameras_per_object = 2
price_per_camera = 3_000  # руб./мес., допущение, не замер рынка

monthly_revenue = objects * cameras_per_object * price_per_camera
annual_revenue = monthly_revenue * 12

COSTS = pd.DataFrame(
    [
        {"item": "установка и выезд", "monthly_rub": None, "note": ""},
        {"item": "мини-ПК / сервер", "monthly_rub": None, "note": ""},
        {"item": "поддержка", "monthly_rub": None, "note": ""},
        {"item": "разработка", "monthly_rub": None, "note": ""},
        {"item": "продажи", "monthly_rub": None, "note": ""},
        {"item": "налоги", "monthly_rub": None, "note": ""},
        {"item": "лицензии ПО", "monthly_rub": None, "note": "Ultralytics: учесть AGPL-3.0 или коммерческую лицензию"},
    ]
)

COMPETITORS = pd.DataFrame(
    [
        {"name": "", "price": "", "local_inference": "", "integrations": "", "weak_for_small_site": ""},
        {"name": "", "price": "", "local_inference": "", "integrations": "", "weak_for_small_site": ""},
        {"name": "", "price": "", "local_inference": "", "integrations": "", "weak_for_small_site": ""},
    ]
)

def rub(amount: int) -> str:
    return f"{amount:,}".replace(",", " ")


print(f"Выручка примера: {rub(monthly_revenue)} ₽/мес., {rub(annual_revenue)} ₽/год")
print("Заполните COSTS и COMPETITORS своими источниками. Выручку не называйте прибылью.")
COSTS


### Риски, которые стоит проговорить на защите

| Риск | Что показать вместо оправдания |
|---|---|
| CRNN не обучена | Рабочий MVP на PaddleOCR и отдельный журнал эксперимента CRNN |
| Мелкие номера теряются | Прогон на 960 px, проверка разметки, примеры ошибок |
| Видео нестабильно | Сравнение одного кадра и агрегации |
| Мало GPU | Меньшая выборка, YOLO11n, меньший batch или облачный GPU; записать, на чём учили |
| Нет интерфейса | Сквозной пайплайн в этом ноутбуке и резервное видео |
| Спрос не проверен | Допущения финансового примера и план интервью с 5–10 площадками |

Правовые ограничения: реальные номера и видео — персональные данные; на демо лучше свои или разрешённые материалы; хранение кадров ограничить; лицензии датасетов и моделей проверить до любого коммерческого использования.


## 7. Чеклист готовности

Отмечайте здесь по мере реальных результатов. Презентация по плану: 10–12 слайдов, спич 5–8 минут, три живых сценария и запасное видео.

- [ ] Датасеты загружены, изучены и разделены без утечки кадров
- [ ] EDA: 5–8 графиков и выводы в разделе 1
- [ ] Обучен детектор, метрики лежат в `DETECTION_EXPERIMENTS`
- [ ] OCR подключено и посчитано на отложенном тесте
- [ ] Есть свой эксперимент, желательно CRNN
- [ ] Не меньше трёх сравнений (аугментации, размер входа или модель, OCR, кадры)
- [ ] Пайплайн: детекция → OCR → решение
- [ ] Сценарии А, Б и В заполнены фактическими результатами
- [ ] Разобраны ошибки и ложные разрешения
- [ ] Три конкурента и экономика с расходами
- [ ] Указано оборудование и задержка
- [ ] Питч и резервное демо готовы

Порядок работ из плана: сначала YOLO + PaddleOCR на фотографии и честные метрики, затем видео, затем интерфейс и только после этого коммерческая часть защиты.
